# 05: Fund Holdings Analysis

This notebook visualizes the portfolio composition of a mutual fund, including top holdings, sector allocation, and (if snapshots available) portfolio changes.

In [ ]:
import pandas as pd
import plotly.express as px
import json
from mf_analyser.data.cache import get_holdings, search_cached_schemes
from mf_analyser.analysis.holdings import get_sector_allocation, get_top_holdings
from mf_analyser.config import FUND_CODE_TO_NAME

# 1. Select a fund
scheme_code = '122639' # Parag Parikh Flexi Cap
fund_name = FUND_CODE_TO_NAME.get(scheme_code, f"Scheme {scheme_code}")

# 2. Fetch/Load holdings
data = get_holdings(scheme_code)
df = pd.DataFrame(data['holdings'])

print(f"Analysing: {fund_name}")
print(f"As of Date: {data.get('as_of_date')}")
print(f"Total Holdings: {data['total_holdings']}")

## Sector Allocation

Understanding the thematic exposure of the fund.

In [ ]:
sector_df = get_sector_allocation(data)

fig = px.pie(
    sector_df, 
    values='weightage', 
    names='sector', 
    title=f"Sector Allocation: {fund_name}",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig.update_traces(textinfo='percent+label')
fig.show()

## Top Holdings Concentration

Visualizing the top 15 stocks by weight.

In [ ]:
top_15 = df.head(15).copy()
top_15 = top_15.sort_values("weightage", ascending=True)

fig = px.bar(
    top_15, 
    x='weightage', 
    y='name', 
    color='sector',
    title=f"Top 15 Holdings: {fund_name}",
    labels={'weightage': 'Concentration (%)', 'name': 'Stock Name'},
    text_auto='.2f'
)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

## Asset Class Breakdown

Equity vs. Cash vs. Debt instruments.

In [ ]:
instrument_df = df.groupby("instrument")["weightage"].sum().reset_index()
fig = px.bar(
    instrument_df, 
    x='instrument', 
    y='weightage', 
    title=f"Asset Class Exposure: {fund_name}",
    labels={'weightage': 'Exposure (%)'},
    color='instrument'
)
fig.show()